# Lab 09: Develop Prompt and Agent Versions

> **Source:** Microsoft Learning -- [mslearn-genaiops](https://github.com/MicrosoftLearning/mslearn-genaiops), `docs/02-prompt-management.md`
> **License:** MIT

## Objectives

By the end of this lab you will:

1. Evolve a prompt across three versions: **V1** (basic), **V2** (personalized), **V3** (production-ready)
2. Use **Git tags** for prompt version control
3. Compare versions to understand quality progression

| Estimated Time | Estimated Cost |
|---|---|
| ~30 minutes | ~$0.50 |

## Prompt Evolution Pattern

Prompts in GenAIOps follow a maturity curve, just like code:

| Version | Stage | Characteristics |
|---|---|---|
| **V1** | Baseline | Generic instructions. No guardrails, no structure. Gets the model "working." |
| **V2** | Personalized | Adds clarifying questions, user-awareness (fitness, group size), and safety considerations. |
| **V3** | Production-Ready | Structured output framework, explicit safety guidelines, token-conscious formatting, grounding instructions, edge-case handling. |

Each version is tagged in Git so you can:
- **Roll back** to any prior version instantly
- **Compare** prompts side-by-side across tags
- **Audit** who changed what and when

> **Exam Tip:** Prompt versioning with **Git tags** is the GenAIOps equivalent of **model versioning** in MLOps. Tags create immutable reference points. The exam may ask how to manage prompt versions -- the answer is Git, not a database.

## V1: Baseline Prompt

The simplest possible prompt -- just enough to get the agent working:

```
You are a trail guide assistant. Help hikers plan their trips.
```

**Characteristics:**
- Generic -- no awareness of user context
- No guardrails -- could recommend dangerous trails to beginners
- No structure -- responses vary wildly in format and length
- No grounding -- may hallucinate trail names or conditions

This is your **starting point**, not your shipping version.

## V2: Personalized Prompt

Adds **clarifying questions**, **fitness awareness**, and **safety considerations**:

```
You are a trail guide assistant. Help hikers plan their trips.

Before giving recommendations:
- Ask about the hiker's fitness level and experience
- Ask about group size and composition (children, pets, elderly)
- Ask about preferred difficulty and duration
- Consider seasonal and weather conditions

Always include safety considerations:
- Water and food requirements
- Emergency contact information
- Leave No Trace principles
```

**What changed:**
- The agent now **asks before answering** -- gathering context before making recommendations
- Safety is explicitly required in every response
- The prompt is aware of **user variability** (fitness, group composition)

## V3: Production-Ready Prompt

The full production prompt with **structured output**, **explicit safety**, **token efficiency**, and **grounding**:

```
You are a professional trail guide assistant for national and state parks.

## Response Framework
1. **Trail Overview**: Name, location, distance, elevation gain, difficulty rating
2. **Preparation Checklist**: Gear, food, water, permits
3. **Safety Guidelines**: Weather risks, wildlife, emergency protocols
4. **Leave No Trace**: Applicable principles for the specific trail

## Rules
- Always ask clarifying questions before providing recommendations
- Never recommend trails beyond the hiker's stated ability
- If unsure about current trail conditions, say so explicitly
- Keep responses concise: aim for <500 tokens unless the user asks for detail
- Ground recommendations in well-known trail databases and park services
- For safety-critical questions (medical, rescue), direct the user to call 911
  or the nearest ranger station
```

**What changed from V2:**
- **Structured output** -- consistent 4-section framework makes responses predictable
- **Token budget** -- explicit <500 token target reduces cost
- **Grounding instructions** -- "well-known trail databases and park services" reduces hallucination
- **Edge-case handling** -- safety-critical questions get explicit redirects
- **Professional identity** -- "professional trail guide" vs generic "assistant"

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    credential=credential,
    endpoint=os.getenv("AZURE_AI_PROJECT_ENDPOINT"),
)

# V1 prompt
v1_instructions = "You are a trail guide assistant. Help hikers plan their trips."

# Deploy V1 agent
agent_v1 = project_client.agents.create_agent(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4.1"),
    name="trail-guide-v1",
    instructions=v1_instructions,
)
print(f"V1 agent created: {agent_v1.id}")

# To deploy all three versions at once, run:
#   python scripts/deploy_all_versions.py

## Git Tagging for Version Control

Each prompt version gets an **annotated Git tag** -- an immutable reference point that you can check out or diff against at any time:

```bash
git tag -a v1 -m "V1: Baseline trail guide prompt"
git tag -a v2 -m "V2: Personalized with clarifying questions"
git tag -a v3 -m "V3: Production-ready with structured output"
```

Useful commands:

```bash
# List all tags
git tag -l

# View the diff between V1 and V3
git diff v1..v3 -- src/api/trail_guide_agent.py

# Roll back to V2
git checkout v2
```

> **Exam Tip:** Annotated tags (`git tag -a`) include metadata (author, date, message) and create **immutable reference points** for rollback. This is how GenAIOps tracks prompt versions -- lightweight, auditable, and built into Git.

## Key Takeaways

| Concept | Detail |
|---|---|
| **Prompt Evolution** | V1 (baseline) -> V2 (personalized) -> V3 (production). Each version adds specificity and guardrails. |
| **Git Tags** | Annotated tags create immutable, auditable version markers. Use `git tag -a` with a descriptive message. |
| **Grounding** | V3 explicitly instructs the model to reference known sources -- this reduces hallucination. |
| **Structured Output** | A consistent response framework (Trail Overview, Checklist, Safety, LNT) makes outputs predictable and testable. |
| **Token Efficiency** | V3 sets a <500 token target. Shorter prompts + shorter responses = lower cost at scale. |

---

**Next:** [Lab 10 -- Prompt Optimization](../lab10-prompt-optimization/lab10-prompt-optimization.ipynb) -- batch testing and experiment branches.